In [3]:
import numpy as np
import pandas as pd

# 1. CONFIGURATION
np.random.seed(42)
N_USERS = 10
N_DAYS = 30

# Dynamic Threshold
THRESH_START = 2.5
THRESH_END = 1.8

# EMA config
EMA_SPAN = 20

# GENERATE USERS

def generate_users(n_users):
    user_ids = [f"User_{i:02d}" for i in range(1, n_users + 1)]
    ages = np.random.randint(18, 80, n_users)
    bmis = np.random.uniform(18.0, 35.0, n_users)
    genders = np.random.choice([0, 1], n_users)

    return pd.DataFrame({'User_ID': user_ids, 'Age': ages, 'BMI': bmis, 'Gender': genders})

# GENERATE TIME SERIES DATA
def generate_time_series(profiles, n_days):
    records = []

    for _, user in profiles.iterrows():
        uid = user['User_ID']
        age, bmi = user['Age'], user['BMI']

        base_hr = 50 + ((age - 18) * 0.15) + ((bmi - 18) * 1.0)
        base_bpsys = 100 + (age * 0.3) + (bmi * 0.8)
        base_bpdia = 65 + (age * 0.15) + (bmi * 0.5)
        base_sugar = 75 + (bmi * 0.7) + (age * 0.1)
        base_water = 2000
        base_steps = 10000 - (age * 30) - (bmi * 50)

        for day in range(1, n_days + 1):
            anomaly_mult = 3 if np.random.rand() < 0.05 else 1 # 5% chance of a spike

            records.append({
                'User_ID': uid, 'Day': day,
                'HR_avg': base_hr + np.random.normal(0, 4 * anomaly_mult),
                'BP_sys': base_bpsys + np.random.normal(0, 5 * anomaly_mult),
                'BP_dia': base_bpdia + np.random.normal(0, 4 * anomaly_mult),
                'Blood_Sugar': base_sugar + np.random.normal(0, 8 * anomaly_mult),
                'Water_intake': base_water + np.random.normal(0, 400 * anomaly_mult),
                'Steps': base_steps + np.random.normal(0, 1500 * anomaly_mult)
            })

    return pd.DataFrame(records)

# EXECUTION
profiles_df = generate_users(N_USERS)
df = pd.merge(generate_time_series(profiles_df, N_DAYS), profiles_df, on='User_ID')
df.sort_values(by=['User_ID', 'Day'], inplace=True)

df['Dynamic_Threshold'] = THRESH_START - ((THRESH_START - THRESH_END) * (df['Day'] - 1) / (N_DAYS - 1))

dynamic_features = ['HR_avg', 'BP_sys', 'BP_dia', 'Blood_Sugar', 'Water_intake', 'Steps']
z_cols = []

for col in dynamic_features:
    mean_col = f'{col}_EMA_Mean'
    df[mean_col] = df.groupby('User_ID')[col].transform(lambda x: x.ewm(span=EMA_SPAN, min_periods=1).mean())

    std_col = f'{col}_EMA_Std'
    df[std_col] = df.groupby('User_ID')[col].transform(lambda x: x.ewm(span=EMA_SPAN, min_periods=1).std())

    df[std_col] = df.groupby('User_ID')[std_col].bfill().replace(0, 0.1)

    z_col_name = f'{col}_Z'

    df[z_col_name] = ((df[col] - df[mean_col]) / df[std_col]).abs()
    z_cols.append(z_col_name)

df['Max_Z_Score'] = df[z_cols].max(axis=1)
df['Is_Anomaly'] = df['Max_Z_Score'] > df['Dynamic_Threshold']

#Printing

print(f"Dataset Shape: {df.shape} ({N_USERS} users × {N_DAYS} days)")
print(f"Threshold Decay: Day 1 ({THRESH_START}) -> Day 30 ({THRESH_END})")
print("-" * 55)

total_records = len(df)
total_anomalies = df['Is_Anomaly'].sum()
print(f"TOTAL ANOMALIES: {total_anomalies} out of {total_records} days")
print(f"OVERALL ANOMALY RATE: {(total_anomalies / total_records) * 100:.2f}%\n")

print("--- ANOMALIES PER USER ---")
for uid in sorted(df['User_ID'].unique()):
    user_data = df[df['User_ID'] == uid]
    anomalous_days = user_data[user_data['Is_Anomaly']]

    anomaly_count = len(anomalous_days)
    user_percentage = (anomaly_count / N_DAYS) * 100

    if anomaly_count > 0:
        day_info = [f"Day {r['Day']} (Thresh: {r['Dynamic_Threshold']:.2f})" for _, r in anomalous_days.iterrows()]
        print(f"[{uid}] {anomaly_count} Anomalies ({user_percentage:.1f}%) | {', '.join(day_info)}")
    else:
        print(f"[{uid}] 0 Anomalies (0.0%) | User remained stable.")

Dataset Shape: (300, 32) (10 users × 30 days)
Threshold Decay: Day 1 (2.5) -> Day 30 (1.8)
-------------------------------------------------------
TOTAL ANOMALIES: 8 out of 300 days
OVERALL ANOMALY RATE: 2.67%

--- ANOMALIES PER USER ---
[User_01] 1 Anomalies (3.3%) | Day 30 (Thresh: 1.80)
[User_02] 1 Anomalies (3.3%) | Day 23 (Thresh: 1.97)
[User_03] 0 Anomalies (0.0%) | User remained stable.
[User_04] 1 Anomalies (3.3%) | Day 13 (Thresh: 2.21)
[User_05] 1 Anomalies (3.3%) | Day 27 (Thresh: 1.87)
[User_06] 1 Anomalies (3.3%) | Day 25 (Thresh: 1.92)
[User_07] 1 Anomalies (3.3%) | Day 23 (Thresh: 1.97)
[User_08] 1 Anomalies (3.3%) | Day 20 (Thresh: 2.04)
[User_09] 0 Anomalies (0.0%) | User remained stable.
[User_10] 1 Anomalies (3.3%) | Day 22 (Thresh: 1.99)
